# **作业 6 - 生成对抗网络 (GAN)**

这是李宏毅老师机器学习课程作业 6 的示例代码。

本次作业要求构建一个生成对抗网络，用于动漫人脸生成。

## 环境配置

### 工作目录

In [ ]:
# Kaggle 工作目录（输出保存位置）
workspace_dir = "/kaggle/working"

### 导入依赖包

In [ ]:
import os

# NCCL 兼容性修复 (Kaggle T4 双卡常见此问题)
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
import glob
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch import optim
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader

from qqdm.notebook import qqdm

### 数据集
Kaggle 数据集：https://www.kaggle.com/datasets/jonescrystal/crypko

在 Kaggle Notebook 中，添加该数据集后会自动下载到 `/kaggle/input/crypko/`。

数据集内包含 `faces/` 子目录，包含所有动漫人脸图像。

### 解压下载的文件
解压后的目录结构如下：
```
faces/
├── 1.jpg
├── 2.jpg
├── 3.jpg
...
```

## 随机种子
设置随机种子为固定值，确保实验可复现。

In [ ]:
def same_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


same_seeds(2026)

# 设备配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
print(f"设备: {device}, GPU 数量: {n_gpu}")

## 数据集
1. 将图像缩放至 (64, 64)
2. 将像素值从 [0, 1] 线性映射到 [-1, 1]

各种 transform 的详细说明请参考 [PyTorch 官方文档](https://pytorch.org/vision/stable/transforms.html)。

In [ ]:
class CrypkoDataset(Dataset):
    def __init__(self, fnames, transform):
        self.transform = transform
        self.fnames = fnames
        self.num_samples = len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        # 1. 加载图像
        img = torchvision.io.read_image(fname)
        # 2. 使用 torchvision 缩放并归一化图像
        img = self.transform(img)
        return img

    def __len__(self):
        return self.num_samples


def get_dataset(root):
    fnames = glob.glob(os.path.join(root, "*"))
    # 1. 将图像缩放至 (64, 64)
    # 2. 将 [0, 1] 线性映射到 [-1, 1]
    compose = [
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
    transform = transforms.Compose(compose)
    dataset = CrypkoDataset(fnames, transform)
    return dataset

### 展示部分图像
注意：当前像素值范围为 [-1, 1]，需要将其平移至有效范围 [0, 1] 才能正确显示。

In [ ]:
# Kaggle: 数据集在 /kaggle/input/crypko/faces/
dataset = get_dataset("/kaggle/input/datasets/jonescrystal/crypko/faces")

images = [dataset[i] for i in range(16)]
grid_img = torchvision.utils.make_grid(images, nrow=4)
plt.figure(figsize=(10, 10))
plt.imshow(grid_img.permute(1, 2, 0))
plt.show()

[-1,1] 反向映射到 [0,1] 是通过 $\frac{x + 1}{2}$ 得到

In [ ]:
images = [(dataset[i] + 1) / 2 for i in range(16)]
grid_img = torchvision.utils.make_grid(images, nrow=4)
plt.figure(figsize=(10, 10))
plt.imshow(grid_img.permute(1, 2, 0))
plt.show()

## 模型
这里使用 DCGAN (Deep Convolutional Generative Adversarial Network) 作为模型结构，可以自由修改模型结构。

注意：输入/输出形状中的 `N` 代表 batch size。

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        m.weight.data.normal_(0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)


class Generator(nn.Module):
    """
    Input shape: (N, in_dim)
    Output shape: (N, 3, 64, 64)
    """

    def __init__(self, in_dim, dim=64):
        super().__init__()

        def dconv_bn_relu(in_dim, out_dim):
            return nn.Sequential(
                nn.ConvTranspose2d(
                    in_dim, out_dim, 5, 2, padding=2, output_padding=1, bias=False
                ),
                nn.BatchNorm2d(out_dim),
                nn.ReLU(),
            )

        self.l1 = nn.Sequential(
            nn.Linear(in_dim, dim * 8 * 4 * 4, bias=False),
            nn.BatchNorm1d(dim * 8 * 4 * 4),
            nn.ReLU(),
        )
        self.l2_5 = nn.Sequential(
            dconv_bn_relu(dim * 8, dim * 4),
            dconv_bn_relu(dim * 4, dim * 2),
            dconv_bn_relu(dim * 2, dim),
            nn.ConvTranspose2d(dim, 3, 5, 2, padding=2, output_padding=1),
            nn.Tanh(),
        )
        self.apply(weights_init)

    def forward(self, x):
        y = self.l1(x)
        y = y.view(y.size(0), -1, 4, 4)
        y = self.l2_5(y)
        return y


class Discriminator(nn.Module):
    """
    Input shape: (N, 3, 64, 64)
    Output shape: (N, )
    """

    def __init__(self, in_dim, dim=64):
        super(Discriminator, self).__init__()

        def conv_bn_lrelu(in_dim, out_dim):
            return nn.Sequential(
                nn.Conv2d(in_dim, out_dim, 5, 2, 2),
                nn.LeakyReLU(0.2),
            )

        """ WGAN: 不使用 Sigmoid 激活 """
        self.ls = nn.Sequential(
            nn.Conv2d(in_dim, dim, 5, 2, 2),
            nn.LeakyReLU(0.2),
            conv_bn_lrelu(dim, dim * 2),
            conv_bn_lrelu(dim * 2, dim * 4),
            conv_bn_lrelu(dim * 4, dim * 8),
            nn.Conv2d(dim * 8, 1, 4),
        )
        self.apply(weights_init)

    def forward(self, x):
        y = self.ls(x)
        y = y.view(-1)
        return y

## 训练

### 初始化
- 超参数
- 模型
- 优化器
- 数据加载器

In [ ]:
# 训练超参数
batch_size = 512
z_dim = 100
lr = 1e-4
n_epoch = 200

# 固定采样噪声（用于可视化训练过程）
z_sample = torch.randn(100, z_dim).to(device)

log_dir = os.path.join(workspace_dir, 'logs')
ckpt_dir = os.path.join(workspace_dir, 'checkpoints')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

# 模型
G = Generator(in_dim=z_dim).to(device)
D = Discriminator(3).to(device)

# 多 GPU 支持
if n_gpu > 1:
    G = nn.DataParallel(G)
    D = nn.DataParallel(D)
    print(f'使用 {n_gpu} 张 GPU 并行训练')

G.train()
D.train()

# WGAN-GP 优化器 (Adam, betas=(0.0, 0.9) 来自原论文)
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.0, 0.9))
opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))

# 余弦退火学习率
scheduler_D = optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=n_epoch)
scheduler_G = optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=n_epoch)

# 数据加载器
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# 断点续训
start_epoch = 0
steps = 0
ckpt_path = os.path.join(ckpt_dir, 'checkpoint.pth')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and 'G' in ckpt:
        G.load_state_dict(ckpt['G'])
        D.load_state_dict(ckpt['D'])
        opt_G.load_state_dict(ckpt['opt_G'])
        opt_D.load_state_dict(ckpt['opt_D'])
        start_epoch = ckpt['epoch']
        steps = ckpt['steps']
        # 快进 scheduler 到恢复的 epoch
        for _ in range(start_epoch):
            scheduler_D.step()
            scheduler_G.step()
        print(f'从检查点恢复训练: Epoch {start_epoch}, Steps {steps}')
    else:
        print('检查点格式不兼容，从头开始训练')
else:
    print('未找到检查点，从头开始训练')

### 训练循环
定期保存部分生成图像，用于监控生成器的当前表现，并定期记录模型检查点。

In [ ]:
def gradient_penalty(D, real, fake, device):
    """WGAN-GP 梯度惩罚：约束 D 的梯度范数接近 1"""
    batch_size = real.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolates = alpha * real + (1 - alpha) * fake
    interpolates.requires_grad_(True)

    d_interpolates = D(interpolates)

    grad_outputs = torch.ones_like(d_interpolates)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    gp = ((gradient_norm - 1) ** 2).mean()
    return gp

In [ ]:
n_critic = 5  # WGAN-GP: 每训练 5 次 D 训练 1 次 G
lambda_gp = 10  # 梯度惩罚系数

for e, epoch in enumerate(range(start_epoch, n_epoch)):
    progress_bar = qqdm(dataloader)
    for batch_idx, data in enumerate(progress_bar):
        imgs = data.to(device, non_blocking=True)
        bs = imgs.size(0)

        # ============================================
        #  训练判别器 D
        # ============================================
        z = torch.randn(bs, z_dim, device=device)

        opt_D.zero_grad()
        f_imgs = G(z)
        # WGAN 损失: D 希望最大化 D(real) - D(fake)
        loss_D = -torch.mean(D(imgs)) + torch.mean(D(f_imgs.detach()))
        # 梯度惩罚
        gp = gradient_penalty(D, imgs, f_imgs.detach(), device)
        loss_D = loss_D + lambda_gp * gp
        loss_D.backward()
        opt_D.step()

        # ============================================
        #  训练生成器 G (每 n_critic 步训练一次)
        # ============================================
        if steps % n_critic == 0:
            z = torch.randn(bs, z_dim, device=device)

            opt_G.zero_grad()
            f_imgs = G(z)
            # WGAN 损失: G 希望最大化 D(fake)
            loss_G = -torch.mean(D(f_imgs))
            loss_G.backward()
            opt_G.step()

        steps += 1

        progress_bar.set_infos({
            'Loss_D': round(loss_D.item(), 4),
            'Loss_G': round(loss_G.item(), 4),
            'Epoch': e + 1,
            'Step': steps,
            'LR': f'{scheduler_D.get_last_lr()[0]:.2e}',
        })

    # 每 10 个 epoch 保存一次生成样本
    if (e + 1) % 10 == 0 or e == 0:
        G.eval()
        with torch.no_grad():
            f_imgs_sample = (G(z_sample) + 1) / 2.0
        filename = os.path.join(log_dir, f'Epoch_{epoch+1:03d}.jpg')
        torchvision.utils.save_image(f_imgs_sample, filename, nrow=10)
        print(f' | 保存样本到 {filename}')

        grid_img = torchvision.utils.make_grid(f_imgs_sample.cpu(), nrow=10)
        plt.figure(figsize=(10, 10))
        plt.imshow(grid_img.permute(1, 2, 0))
        plt.show()
        G.train()

    if (e + 1) % 5 == 0 or e == 0:
        # 保存模型 (处理 DataParallel 前缀)
        g_state = G.module.state_dict() if hasattr(G, 'module') else G.state_dict()
        d_state = D.module.state_dict() if hasattr(D, 'module') else D.state_dict()
        # 完整检查点（用于断点续训）
        torch.save({
            'epoch': e + 1,
            'steps': steps,
            'G': g_state,
            'D': d_state,
            'opt_G': opt_G.state_dict(),
            'opt_D': opt_D.state_dict(),
        }, os.path.join(ckpt_dir, 'checkpoint.pth'))
        # 单独模型文件（用于推理）
        torch.save(g_state, os.path.join(ckpt_dir, 'G.pth'))
        torch.save(d_state, os.path.join(ckpt_dir, 'D.pth'))
        print(f' | Epoch {epoch+1} 检查点已保存')

    scheduler_D.step()
    scheduler_G.step()

## 推理
使用训练好的模型生成动漫人脸！

### 加载模型

In [ ]:
G = Generator(z_dim).to(device)
G.load_state_dict(torch.load(os.path.join(ckpt_dir, 'G.pth'), map_location=device))
G.eval()

### 生成并展示部分图像

In [ ]:
# 生成 1000 张图像
n_output = 1000
z_sample = torch.randn(n_output, z_dim, device=device)

with torch.no_grad():
    imgs_sample = (G(z_sample) + 1) / 2.0

log_dir = os.path.join(workspace_dir, 'logs')
filename = os.path.join(log_dir, 'result.jpg')
torchvision.utils.save_image(imgs_sample, filename, nrow=10)

# 展示 32 张
grid_img = torchvision.utils.make_grid(imgs_sample[:32].cpu(), nrow=10)
plt.figure(figsize=(10, 10))
plt.imshow(grid_img.permute(1, 2, 0))
plt.show()

### 使用 **tar** 压缩生成的图像

In [ ]:
# 保存生成的图像
os.makedirs('output', exist_ok=True)
for i in range(1000):
    torchvision.utils.save_image(imgs_sample[i], f'output/{i+1}.jpg')
  
# 压缩图像
%cd output
!tar -zcf ../images.tgz *.jpg
%cd ..